# Реальная лабораторная 01 — Online Retail

Бизнес-данные: товарные строки, возвраты, продажи, клиенты и когорты.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## 1. Контекст и единица наблюдения

Набор содержит строки товарных позиций внутри счетов, а не строки клиентов и не обязательно один ряд на заказ. У одного InvoiceNo много StockCode. Отдельно различай строку, счёт, клиента и товар. Период наблюдения ограничен 2010-12 — 2011-12; это исторический кейс, не современный рынок.

In [ ]:
df = load_real("online_retail")
print("shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

## 2. Первичный аудит

Сначала оцени пропуски, уникальность ключей и допустимые диапазоны. Не предполагай, что `CustomerID` заполнен в каждой строке: измерь фактическую полноту. `InvoiceNo` и `StockCode` — идентификаторы, а не количественные признаки.

In [ ]:
required = {"InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"}
missing_cols = required - set(df.columns)
assert not missing_cols, f"Не найдены столбцы: {missing_cols}"
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "n_unique": df.nunique(dropna=True)
})
display(quality.sort_values("missing_pct", ascending=False))
print("Полные дубликаты:", df.duplicated().sum())
print("Дубли строк по InvoiceNo + StockCode:", df.duplicated(["InvoiceNo","StockCode"]).sum())

## 3. Возвраты, отмены и денежный поток

Префикс `C` в InvoiceNo обозначает отмену. Отрицательные количества могут быть возвратами; нулевая/отрицательная цена требует отдельной проверки. Не смешивай gross sales, net sales и возвраты в одной метрике.

In [ ]:
df["is_cancel_invoice"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
df["line_value"] = df["Quantity"] * df["UnitPrice"]
print("Отменённые строки:", int(df["is_cancel_invoice"].sum()))
print("Отрицательное количество:", int((df["Quantity"] < 0).sum()))
print("Нулевая/отрицательная цена:", int((df["UnitPrice"] <= 0).sum()))
display(df.groupby("is_cancel_invoice")["line_value"].agg(["count","sum","median"]))

## 4. Временная структура и выручка

Построй временной ряд по дням и проверь, нет ли разрывов, резких всплесков и эффектов календаря. Для описания продаж используй явное правило фильтрации, а не просто `line_value.sum()`.

In [ ]:
sales = df.loc[
    (~df["is_cancel_invoice"]) & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)
].copy()
sales["date"] = sales["InvoiceDate"].dt.floor("D")
daily = sales.groupby("date").agg(
    revenue_gbp=("line_value","sum"),
    invoice_count=("InvoiceNo","nunique"),
    line_count=("InvoiceNo","size"),
    customer_count=("CustomerID","nunique")
)
display(daily.head())
ax = daily["revenue_gbp"].plot(figsize=(13,4), title="Daily gross sales (positive, non-cancelled lines)")
ax.set_ylabel("GBP"); plt.show()

## 5. Клиенты и RFM как описательная сегментация

RFM рассчитывается на фиксированную дату отсечения. Это описательная характеристика прошлого, а не готовая модель будущей ценности. Укажи snapshot date и не включай возвраты в positive-sales RFM без отдельного обоснования.

In [ ]:
snapshot = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
customer = sales.dropna(subset=["CustomerID"]).groupby("CustomerID").agg(
    last_purchase=("InvoiceDate","max"),
    frequency=("InvoiceNo","nunique"),
    monetary=("line_value","sum")
)
customer["recency_days"] = (snapshot - customer["last_purchase"]).dt.days
display(customer.describe().T)
fig, axes = plt.subplots(1,3,figsize=(15,3.8))
for col, ax in zip(["recency_days","frequency","monetary"], axes):
    values = customer[col].dropna().to_numpy()
    if len(values) == 0:
        ax.text(.5,.5,"No data after filters",ha="center",va="center")
    elif np.unique(values).size == 1:
        ax.axvline(values[0], linewidth=2)
        ax.set_title(f"{col} (constant in this sample)")
    else:
        ax.hist(values, bins=min(40, max(5, int(np.sqrt(len(values))))), edgecolor="white")
        ax.set_title(col)
plt.tight_layout(); plt.show()

## 6. Когорты покупателей

Когорта определяется месяцем первой покупки. Retention — доля клиентов исходной когорты, совершивших покупку в последующий месяц. Последние когорты имеют меньше времени наблюдения: неполные будущие периоды нельзя считать нулевым retention.

In [ ]:
cohort_data = sales.dropna(subset=["CustomerID"]).copy()
cohort_data["invoice_month"] = cohort_data["InvoiceDate"].dt.to_period("M")
first_month = cohort_data.groupby("CustomerID")["invoice_month"].min().rename("cohort_month")
cohort_data = cohort_data.join(first_month, on="CustomerID")
cohort_data["cohort_index"] = (
    (cohort_data["invoice_month"].dt.year - cohort_data["cohort_month"].dt.year) * 12
    + (cohort_data["invoice_month"].dt.month - cohort_data["cohort_month"].dt.month)
)
cohort_counts = cohort_data.groupby(["cohort_month","cohort_index"])["CustomerID"].nunique().unstack()
cohort_sizes = cohort_counts.iloc[:,0]
retention = cohort_counts.divide(cohort_sizes, axis=0)
plt.figure(figsize=(13,6)); sns.heatmap(retention, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1)
plt.title("Customer cohort retention (blank cells = not observed)"); plt.show()

## 7. Самостоятельные задания

1. Сравни страны по числу клиентов, числу чеков и выручке. Почему ранжирование меняется?
2. Найди товары с высокой долей возвратов. Как определена доля и какой знаменатель выбран?
3. Сравни медианный чек и средний чек. Что говорит различие об асимметрии?
4. Проведи чувствительный анализ: как меняются RFM-сегменты при исключении верхнего 1% monetary?
5. Напиши абзац с ограничениями: оптовая природа части покупателей, исторический период, неполные CustomerID и отсутствие данных о себестоимости.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.